# ProstT5 Speculative Decoding — enc-CNN + Profile HMM Drafters (3Di → AA)

This notebook implements **actual speculative decoding** for ProstT5 inverse folding, using the enc-CNN as a fast drafter.

**Prerequisites:** Run `prostT5_baseline_performance.ipynb` first to generate baseline timings and α predictions.

## What this notebook does

1. **Custom speculative decoding loop** — Leviathan’s algorithm from scratch (greedy + stochastic)
2. **HuggingFace `assistant_model` wrapper** — CNN wrapped as a compatible assistant for `model.generate()`
3. **Benchmark** — Sweep draft lengths K ∈ {1, 3, 5, 8, 16}, measure real speedup
4. **Compare** — Predicted speedup (from α) vs. measured speedup

Key property: greedy speculative decoding produces **exactly the same output** as plain enc-dec greedy generation.

References:
- Leviathan et al. (2023). Fast Inference from Transformers via Speculative Decoding. ICML.
- HuggingFace blog: Assisted Generation (https://huggingface.co/blog/assisted-generation)

In [ ]:
%pip install tiktoken sentencepiece torch
%pip install 'accelerate>=0.26.0'
%pip install "transformers==4.46.3" "protobuf>=3.20" sentencepiece

In [ ]:
#@title Google Drive mount + env vars. { display-mode: "form" }
import os

os.environ["USE_TF"] = "0"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    DRIVE_ROOT = "/content/drive/MyDrive/prostT5_benchmarks"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    USE_DRIVE = True
    print(f"Google Drive mounted. HF cache -> {os.environ['HF_HOME']}")
except (ImportError, ModuleNotFoundError):
    print("Not on Colab; using local paths.")
    DRIVE_ROOT = None
    USE_DRIVE = False

In [ ]:
#@title Imports. { display-mode: "form" }
import os, time, json, gc, statistics, pickle
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM, GenerationConfig

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"torch={torch.__version__}  device={device}  cuda={torch.cuda.is_available()}")
if device.type == "cpu":
    print("WARNING: running on CPU — timings will not reflect realistic GPU latency.")

In [ ]:
#@title Configuration. { display-mode: "form" }
PROSTT5_NAME = "Rostlab/ProstT5_fp16"
NOTEBOOK_DIR = Path(".").resolve()

if USE_DRIVE:
    PROJECT_DIR = Path(DRIVE_ROOT)
else:
    PROJECT_DIR = NOTEBOOK_DIR

CNN_CKPT = PROJECT_DIR / "model.pt"
if not CNN_CKPT.exists():
    for candidate in [NOTEBOOK_DIR / "model.pt",
                      NOTEBOOK_DIR / "cnn_chkpnt_AA_CNN" / "model.pt"]:
        if candidate.exists():
            CNN_CKPT = candidate
            break

RESULTS_DIR = NOTEBOOK_DIR / "spec_decode_results"
RESULTS_DIR.mkdir(exist_ok=True)

if USE_DRIVE:
    CHECKPOINT_DIR = Path(DRIVE_ROOT) / "spec_decode_checkpoints"
else:
    CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Draft lengths to sweep
K_VALUES = [1, 3, 5, 8, 16]

# Timing protocol
NUM_WARMUP = 2
NUM_REPEATS = 3
USE_FP16 = True

# AA vocabulary (alphabetical, matches CNN output classes)
AA_LETTERS = "ACDEFGHIKLMNPQRSTVWY"
assert len(AA_LETTERS) == 20

print(f"CNN checkpoint: {CNN_CKPT}  (exists: {CNN_CKPT.exists()})")
print(f"Results dir: {RESULTS_DIR}")
print(f"K values to sweep: {K_VALUES}")

In [ ]:
#@title Load ProstT5 + CNN. { display-mode: "form" }
tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)

dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(
    PROSTT5_NAME, low_cpu_mem_usage=True, torch_dtype=dtype,
).to(device).eval()
if dtype == torch.float16:
    model = model.half()

encoder = model.get_encoder()

class AACNN(nn.Module):
    def __init__(self, num_tokens=20, hidden=32, in_dim=1024):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Conv2d(in_dim, hidden, kernel_size=(7, 1), padding=(3, 0)),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Conv2d(hidden, num_tokens, kernel_size=(7, 1), padding=(3, 0)),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1).unsqueeze(-1)
        x = self.classifier(x)
        return x.squeeze(-1).permute(0, 2, 1)

cnn = AACNN(num_tokens=20).to(device).eval()
ckpt = torch.load(CNN_CKPT, map_location=device, weights_only=False)
state_dict = ckpt.get("state_dict", ckpt)
cnn.load_state_dict(state_dict, strict=True)

# Pre-compute AA token IDs in ProstT5 vocabulary
AA_TOKEN_IDS = [tokenizer.convert_tokens_to_ids(aa) for aa in AA_LETTERS]
AA_TOKEN_ID_TO_CNN_IDX = {tid: i for i, tid in enumerate(AA_TOKEN_IDS)}
CNN_IDX_TO_TOKEN_ID = {i: tid for i, tid in enumerate(AA_TOKEN_IDS)}

# Decoder start token
DECODER_START_TOKEN_ID = model.config.decoder_start_token_id

print(f"ProstT5 loaded. params={sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"CNN loaded. params={sum(p.numel() for p in cnn.parameters()):,}")
print(f"Decoder start token ID: {DECODER_START_TOKEN_ID}")
print(f"AA token IDs (first 5): {list(zip(AA_LETTERS[:5], AA_TOKEN_IDS[:5]))}")

In [ ]:
#@title Load test set + baseline results. { display-mode: "form" }
import sys
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(NOTEBOOK_DIR))

try:
    from test_set_100 import TEST_SET, TEST_IDS, BACKUP_IDS
    print(f"Loaded {len(TEST_SET)} proteins from test_set_100.py")
except ImportError:
    print("test_set_100.py not found; using fallback.")
    TEST_IDS = [
        "P62990", "P01308", "P02798", "P62944", "P01542",
        "P43408", "P0AFL6", "P0ACJ8", "P0A988", "P0A7Y4",
    ]

# Load baseline checkpoint (has test_set with 3Di sequences)
baseline_ckpt_path = CHECKPOINT_DIR.parent / "checkpoints" / "checkpoint_latest.pkl"
if not baseline_ckpt_path.exists():
    baseline_ckpt_path = RESULTS_DIR.parent / "benchmark_results" / "checkpoints" / "checkpoint_latest.pkl"

# Try loading the test set from the FASTA files built by notebook 1
DATA_DIR = NOTEBOOK_DIR / "benchmark_data"
TEST_FASTA = DATA_DIR / "test_set_3Di.fasta"
TEST_AA_FASTA = DATA_DIR / "test_set_AA.fasta"

test_set = {}
if TEST_FASTA.exists() and TEST_AA_FASTA.exists():
    cur = None
    for line in TEST_AA_FASTA.read_text().splitlines():
        if line.startswith(">"):
            cur = line[1:].split()[0]
            test_set.setdefault(cur, {})["aa"] = ""
        elif cur:
            test_set[cur]["aa"] += line.strip()
    cur = None
    for line in TEST_FASTA.read_text().splitlines():
        if line.startswith(">"):
            cur = line[1:].split()[0]
            test_set.setdefault(cur, {})["3di"] = ""
        elif cur:
            test_set[cur]["3di"] += line.strip().lower()
    for uid, rec in test_set.items():
        rec["length"] = len(rec.get("3di", ""))
    print(f"Loaded test set from FASTA: {len(test_set)} proteins")
else:
    print("WARNING: FASTA files not found. Run baseline notebook first.")
    print(f"Expected at: {TEST_FASTA}")

# Load alpha predictions from baseline (if available)
alpha_predictions = {}
baseline_ckpt = None
for ckpt_path in [CHECKPOINT_DIR.parent / "checkpoints" / "checkpoint_latest.pkl",
                  Path(DRIVE_ROOT) / "checkpoints" / "checkpoint_latest.pkl" if DRIVE_ROOT else Path("/dev/null")]:
    if ckpt_path.exists():
        with open(ckpt_path, "rb") as f:
            baseline_ckpt = pickle.load(f)
        alpha_predictions = baseline_ckpt.get("alpha_results", {})
        print(f"Loaded baseline checkpoint: {len(alpha_predictions)} proteins with alpha data")
        break

if not alpha_predictions:
    print("No alpha predictions found (baseline not yet complete). Will skip comparison plots.")

## Part 1 — Custom Speculative Decoding Implementation

Implements Leviathan’s algorithm from scratch. Supports greedy (deterministic, output identical to enc-dec) and stochastic (samples from same distribution as enc-dec) modes.

In [ ]:
#@title Helper: format input + reference generation. { display-mode: "form" }

def _format_3di(seq: str) -> str:
    return "<fold2AA> " + " ".join(list(seq.lower()))


def _decode_aa(token_ids: torch.Tensor) -> str:
    s = tokenizer.decode(token_ids, skip_special_tokens=True)
    return "".join(s.split())


@torch.inference_mode()
def generate_reference(three_di: str) -> str:
    """Plain enc-dec greedy generation (the ground truth for correctness checks)."""
    L = len(three_di)
    enc = tokenizer([_format_3di(three_di)], add_special_tokens=True, return_tensors="pt").to(device)
    out = model.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_length=L + 2,
        do_sample=False,
        num_beams=1,
    )
    return _decode_aa(out[0])[:L]

In [ ]:
#@title Core: speculative_decode() — greedy mode. { display-mode: "form" }

@torch.inference_mode()
def speculative_decode_greedy(three_di: str, K: int = 5,
                               get_draft_logits=None,
                               dynamic_k: bool = False,
                               K_init: int = 5,
                               K_min: int = 1,
                               K_max: int = 16) -> dict:
    """
    Speculative decoding with greedy verification.
    Output is EXACTLY the same as plain enc-dec greedy generation.

    Args:
        three_di: lowercase 3Di sequence
        K: fixed draft length (used when dynamic_k=False)
        get_draft_logits: callable(pos, accepted_prefix, encoder_hidden) -> Tensor(k, 20)
                          If None, defaults to CNN drafter.
        dynamic_k: if True, adjust K each step using the heuristic schedule:
                   all accepted -> K+2, any rejection -> K-1 (clamped to [K_min, K_max])
        K_init: starting K for dynamic mode
        K_min: minimum K for dynamic mode
        K_max: maximum K for dynamic mode
    """
    L = len(three_di)

    # 1. Encode (shared, one-time)
    enc_input = tokenizer(
        [_format_3di(three_di)], add_special_tokens=True, return_tensors="pt"
    ).to(device)
    encoder_out = encoder(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
    )
    encoder_hidden = encoder_out.last_hidden_state  # (1, seq_len, 1024)

    # Default to CNN drafter
    if get_draft_logits is None:
        h_cnn = encoder_hidden[:, 1:-1, :]
        cnn_logits_all = cnn(h_cnn.float())[0]  # (L, 20)
        def get_draft_logits(pos, accepted_prefix, enc_hidden):
            k = min(K, L - pos)
            return cnn_logits_all[pos:pos+k]  # (k, 20) — ignores prefix

    # Speculative decoding loop
    generated_token_ids = [DECODER_START_TOKEN_ID]
    accepted_counts = []
    k_history = []
    k_current = K_init if dynamic_k else K
    n_steps = 0

    if device.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    while len(generated_token_ids) - 1 < L:
        pos = len(generated_token_ids) - 1
        k = min(k_current, L - pos)
        if k <= 0:
            break
        k_history.append(k)

        # Draft: get k logits from drafter (shape: k x 20, over AA_LETTERS space)
        draft_logits_20 = get_draft_logits(pos, generated_token_ids[1:], encoder_hidden)
        k = min(k, draft_logits_20.shape[0])  # drafter may return fewer
        draft_cnn_indices = draft_logits_20[:k].argmax(dim=-1)
        draft_token_ids = [CNN_IDX_TO_TOKEN_ID[int(idx)] for idx in draft_cnn_indices]

        # Verify: one decoder forward pass
        verify_input = torch.tensor(
            [generated_token_ids + draft_token_ids], device=device, dtype=torch.long
        )
        dec_out = model(
            encoder_outputs=(encoder_hidden,),
            decoder_input_ids=verify_input,
        )
        verify_logits = dec_out.logits[0]

        # Accept/reject
        n_accepted = 0
        for i in range(k):
            logit_idx = pos + i
            verifier_token = verify_logits[logit_idx].argmax(dim=-1).item()
            if draft_token_ids[i] == verifier_token:
                n_accepted += 1
            else:
                generated_token_ids.extend(draft_token_ids[:n_accepted])
                generated_token_ids.append(verifier_token)
                break
        else:
            generated_token_ids.extend(draft_token_ids)
            if len(generated_token_ids) - 1 < L:
                bonus_logit_idx = pos + k
                if bonus_logit_idx < verify_logits.shape[0]:
                    bonus_token = verify_logits[bonus_logit_idx].argmax(dim=-1).item()
                    generated_token_ids.append(bonus_token)

        accepted_counts.append(n_accepted)
        n_steps += 1

        # Dynamic K adjustment
        if dynamic_k:
            if n_accepted == k:
                k_current = min(k_current + 2, K_max)
            else:
                k_current = max(k_current - 1, K_min)

    if device.type == "cuda":
        torch.cuda.synchronize()
    wall_time = time.perf_counter() - t_start

    output_ids = generated_token_ids[1:]
    output_aa = _decode_aa(torch.tensor(output_ids))[:L]
    tokens_per_step = [(acc + 1) for acc in accepted_counts]

    return {
        "sequence": output_aa,
        "wall_time": wall_time,
        "n_steps": n_steps,
        "n_tokens": len(output_aa),
        "K": K if not dynamic_k else K_init,
        "dynamic_k": dynamic_k,
        "k_history": k_history,
        "accepted_counts": accepted_counts,
        "mean_accepted": np.mean(accepted_counts) if accepted_counts else 0,
        "tokens_per_step": tokens_per_step,
        "mean_tokens_per_step": np.mean(tokens_per_step) if tokens_per_step else 0,
        "acceptance_rate": np.mean(accepted_counts) / np.mean(k_history) if k_history else 0,
    }

In [ ]:
#@title Core: speculative_decode() — stochastic mode. { display-mode: "form" }

@torch.inference_mode()
def speculative_decode_stochastic(three_di: str, K: int = 5,
                                   temperature: float = 1.0) -> dict:
    """
    Speculative decoding with Leviathan's stochastic accept/reject rule.
    Samples from EXACTLY the same distribution as enc-dec with sampling.
    
    Accept token x with probability min(1, p(x)/q(x)).
    On rejection: resample from norm(max(0, p - q)).
    """
    L = len(three_di)
    
    # 1. Encode
    enc_input = tokenizer(
        [_format_3di(three_di)], add_special_tokens=True, return_tensors="pt"
    ).to(device)
    encoder_out = encoder(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
    )
    encoder_hidden = encoder_out.last_hidden_state
    
    # 2. CNN logits
    h_cnn = encoder_hidden[:, 1:-1, :]
    cnn_logits_20 = cnn(h_cnn.float())[0]  # (L_cnn, 20)
    L_cnn = cnn_logits_20.shape[0]
    
    # 3. Speculative decoding loop
    generated_token_ids = [DECODER_START_TOKEN_ID]
    accepted_counts = []
    n_steps = 0
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.perf_counter()
    
    while len(generated_token_ids) - 1 < L:
        pos = len(generated_token_ids) - 1
        k = min(K, L - pos, L_cnn - pos)
        if k <= 0:
            break
        
        # Draft: sample from CNN distribution
        q_logits = cnn_logits_20[pos:pos+k] / max(temperature, 1e-8)  # (k, 20)
        q_probs = F.softmax(q_logits, dim=-1)  # (k, 20)
        draft_cnn_indices = torch.multinomial(q_probs, num_samples=1).squeeze(-1)  # (k,)
        draft_token_ids = [CNN_IDX_TO_TOKEN_ID[int(idx)] for idx in draft_cnn_indices]
        
        # Verify
        verify_input = torch.tensor(
            [generated_token_ids + draft_token_ids], device=device, dtype=torch.long
        )
        dec_out = model(
            encoder_outputs=(encoder_hidden,),
            decoder_input_ids=verify_input,
        )
        verify_logits = dec_out.logits[0]
        
        # Accept/reject with Leviathan's rule
        n_accepted = 0
        for i in range(k):
            logit_idx = pos + i
            
            # Verifier distribution (over full vocab, but we only care about AA tokens)
            p_logits_full = verify_logits[logit_idx] / max(temperature, 1e-8)
            p_probs_full = F.softmax(p_logits_full, dim=-1)
            
            # Get p and q for the drafted token
            drafted_vocab_id = draft_token_ids[i]
            drafted_cnn_idx = int(draft_cnn_indices[i])
            
            p_x = p_probs_full[drafted_vocab_id].item()
            q_x = q_probs[i, drafted_cnn_idx].item()
            
            # Accept with probability min(1, p(x)/q(x))
            accept_prob = min(1.0, p_x / max(q_x, 1e-10))
            r = torch.rand(1).item()
            
            if r < accept_prob:
                n_accepted += 1
            else:
                # Reject: resample from max(0, p - q) restricted to AA tokens
                p_aa = p_probs_full[AA_TOKEN_IDS]  # (20,)
                q_aa = q_probs[i]  # (20,)
                adjusted = torch.clamp(p_aa - q_aa, min=0)
                adjusted_sum = adjusted.sum()
                if adjusted_sum > 1e-10:
                    adjusted = adjusted / adjusted_sum
                    resampled_cnn_idx = torch.multinomial(adjusted, num_samples=1).item()
                else:
                    resampled_cnn_idx = torch.multinomial(p_aa, num_samples=1).item()
                resampled_token_id = AA_TOKEN_IDS[resampled_cnn_idx]
                
                generated_token_ids.extend(draft_token_ids[:n_accepted])
                generated_token_ids.append(resampled_token_id)
                break
        else:
            # All accepted — sample bonus token from verifier
            generated_token_ids.extend(draft_token_ids)
            if len(generated_token_ids) - 1 < L:
                bonus_logit_idx = pos + k
                if bonus_logit_idx < verify_logits.shape[0]:
                    bonus_logits = verify_logits[bonus_logit_idx] / max(temperature, 1e-8)
                    bonus_probs = F.softmax(bonus_logits, dim=-1)
                    bonus_probs_aa = bonus_probs[AA_TOKEN_IDS]
                    bonus_probs_aa = bonus_probs_aa / bonus_probs_aa.sum()
                    bonus_idx = torch.multinomial(bonus_probs_aa, num_samples=1).item()
                    generated_token_ids.append(AA_TOKEN_IDS[bonus_idx])
        
        accepted_counts.append(n_accepted)
        n_steps += 1
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    wall_time = time.perf_counter() - t_start
    
    output_ids = generated_token_ids[1:]
    output_aa = _decode_aa(torch.tensor(output_ids))[:L]
    
    return {
        "sequence": output_aa,
        "wall_time": wall_time,
        "n_steps": n_steps,
        "n_tokens": len(output_aa),
        "K": K,
        "temperature": temperature,
        "accepted_counts": accepted_counts,
        "mean_accepted": np.mean(accepted_counts) if accepted_counts else 0,
        "acceptance_rate": np.mean(accepted_counts) / K if accepted_counts else 0,
    }

In [ ]:
#@title Correctness verification: greedy spec-decode == enc-dec greedy. { display-mode: "form" }

print("Verifying correctness: greedy speculative decoding must produce")
print("EXACTLY the same output as plain enc-dec greedy generation.\n")

# Test on first 5 shortest proteins
test_proteins = sorted(test_set.items(), key=lambda kv: kv[1]["length"])[:5]

all_correct = True
for uid, rec in test_proteins:
    ref = generate_reference(rec["3di"])
    for K in [3, 5, 8]:
        result = speculative_decode_greedy(rec["3di"], K=K)
        match = result["sequence"] == ref
        status = "PASS" if match else "FAIL"
        print(f"  {status}  {uid} L={rec['length']}  K={K}  "
              f"steps={result['n_steps']}  accept_rate={result['acceptance_rate']:.2f}")
        if not match:
            all_correct = False
            # Show first mismatch
            for j, (a, b) in enumerate(zip(ref, result["sequence"])):
                if a != b:
                    print(f"    First mismatch at pos {j}: ref={a} spec={b}")
                    break

print(f"\n{'ALL CORRECT' if all_correct else 'FAILURES DETECTED'}")
if not all_correct:
    print("WARNING: Bug in speculative decoding implementation!")

## Part 2 — Profile HMM Drafter

A Profile HMM models the AA emission probabilities of a protein family at each alignment position. Unlike the CNN, it is **prefix-aware**: after the verifier accepts tokens at positions 1..t, we run the HMM forward algorithm on the accepted prefix to compute the posterior state distribution, then use it to sharpen the emission probabilities at positions t+1..t+K.

**Workflow:**
1. For a query protein, fetch its Pfam family or build an MSA with JackHMMER (against UniRef50)
2. Build a Profile HMM with `hmmbuild` (via `pyhmmer`)
3. Align the query sequence to the HMM to get the position mapping
4. At each spec-decode step: run forward algorithm on accepted prefix → posterior state posteriors → emission logits for next K positions

In [ ]:
#@title Install pyhmmer + fetch MSA for a query protein. { display-mode: "form" }
# pyhmmer is a pure-Python HMMER binding — no system HMMER install needed.
%pip install pyhmmer -q

import pyhmmer
from pyhmmer.easel import SequenceFile, TextSequence, Alphabet
from pyhmmer.plan7 import HMMFile, Builder, Background
import urllib.request, io, tempfile, re
from pathlib import Path

print(f"pyhmmer version: {pyhmmer.__version__}")

HMM_DIR = NOTEBOOK_DIR / "hmm_drafters"
HMM_DIR.mkdir(exist_ok=True)

# AA alphabet for pyhmmer (same 20 standard AAs)
ALPHABET = Alphabet.amino()

# Mapping: pyhmmer residue order -> our AA_LETTERS order
# pyhmmer uses ACDEFGHIKLMNPQRSTVWY (alphabetical) — same as AA_LETTERS
assert AA_LETTERS == "ACDEFGHIKLMNPQRSTVWY"


def fetch_msa_from_pfam(uniprot_id: str, pfam_id: str = None) -> str:
    """Fetch seed MSA for a Pfam family as Stockholm format string.

    If pfam_id is None, first looks up the Pfam family for the given UniProt ID.
    Returns the Stockholm MSA as a string, or None if not available.
    """
    # Step 1: find Pfam family for the protein if not provided
    if pfam_id is None:
        api = f"https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/{uniprot_id}/?format=json"
        try:
            with urllib.request.urlopen(api, timeout=20) as r:
                data = json.loads(r.read())
            entries = data.get("results", [])
            if not entries:
                print(f"  No Pfam entry found for {uniprot_id}")
                return None
            pfam_id = entries[0]["metadata"]["accession"]
            pfam_name = entries[0]["metadata"]["name"]
            print(f"  Found Pfam family: {pfam_id} ({pfam_name})")
        except Exception as e:
            print(f"  Pfam lookup failed for {uniprot_id}: {e}")
            return None

    # Step 2: download seed MSA
    seed_url = f"https://www.ebi.ac.uk/interpro/wwwapi//entry/pfam/{pfam_id}?annotation=alignment:seed&download"
    try:
        with urllib.request.urlopen(seed_url, timeout=30) as r:
            msa_bytes = r.read()
        # Pfam returns gzipped Stockholm
        import gzip
        try:
            msa_str = gzip.decompress(msa_bytes).decode("utf-8")
        except Exception:
            msa_str = msa_bytes.decode("utf-8")
        print(f"  Downloaded seed MSA for {pfam_id}: {len(msa_str)} bytes, "
              f"{msa_str.count('#=GF ID')} alignment(s)")
        return msa_str
    except Exception as e:
        print(f"  MSA download failed for {pfam_id}: {e}")
        return None


def msa_str_to_pyhmmer(msa_str: str) -> pyhmmer.easel.MSA:
    """Parse a Stockholm MSA string into a pyhmmer MSA object."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".sto", delete=False) as f:
        f.write(msa_str)
        tmp_path = f.name
    with pyhmmer.easel.MSAFile(tmp_path, digital=True, alphabet=ALPHABET) as mf:
        msa = mf.read()
    Path(tmp_path).unlink()
    return msa


# Test: fetch MSA for Lysozyme C (P00698, Pfam PF00959 - Phage_lysozyme)
# A small, well-characterized enzyme with deep MSA
print("Testing MSA fetch for Lysozyme C (P00698)...")
test_msa_str = fetch_msa_from_pfam("P00698")
if test_msa_str:
    print("  MSA fetch successful.")

In [ ]:
#@title Build Profile HMM from MSA. { display-mode: "form" }

def build_hmm(msa: pyhmmer.easel.MSA, name: str = "drafter") -> pyhmmer.plan7.HMM:
    """Build a Profile HMM from a pyhmmer MSA object."""
    bg = Background(ALPHABET)
    builder = Builder(ALPHABET)
    hmm, _, _ = builder.build_msa(msa, bg)
    hmm.name = name.encode()
    return hmm


def save_hmm(hmm: pyhmmer.plan7.HMM, path: Path):
    with pyhmmer.plan7.HMMFile.write(str(path)) as f:
        f.write(hmm)


def load_hmm(path: Path) -> pyhmmer.plan7.HMM:
    with pyhmmer.plan7.HMMFile(str(path)) as f:
        return f.read()


def get_hmm_for_protein(uid: str, aa_seq: str, use_cache: bool = True) -> pyhmmer.plan7.HMM | None:
    """Get or build a Profile HMM for a query protein.

    1. Check local cache first.
    2. Fetch Pfam seed MSA via InterPro API.
    3. Build HMM with pyhmmer.
    Returns None if no Pfam family found.
    """
    cache_path = HMM_DIR / f"{uid}.hmm"
    if use_cache and cache_path.exists():
        print(f"  Loading cached HMM for {uid}")
        return load_hmm(cache_path)

    print(f"  Building HMM for {uid}...")
    msa_str = fetch_msa_from_pfam(uid)
    if msa_str is None:
        return None

    try:
        msa = msa_str_to_pyhmmer(msa_str)
        hmm = build_hmm(msa, name=uid)
        save_hmm(hmm, cache_path)
        print(f"  HMM built: M={hmm.M} match states, saved to {cache_path.name}")
        return hmm
    except Exception as e:
        print(f"  HMM build failed for {uid}: {e}")
        return None


def get_hmm_emission_matrix(hmm: pyhmmer.plan7.HMM) -> np.ndarray:
    """Extract match-state emission probabilities from HMM.

    Returns array of shape (M, 20) where M = number of match states,
    values are log probabilities (log-odds or raw log probs).
    """
    M = hmm.M
    # hmm.mat is the match emission probability matrix, shape (M+1, 20)
    # Row 0 is unused (state 0), rows 1..M are match states 1..M
    emissions = np.array(hmm.mat[1:])  # (M, 20), probabilities in log space
    return emissions  # already log probabilities in pyhmmer


# Build HMMs for all proteins that have Pfam entries
print("Building HMMs for test set proteins...")
print("(This fetches Pfam MSAs; may take a few minutes for 100 proteins)")
print("Skipping build for now — will build on demand during spec decoding.")
print(f"HMM cache dir: {HMM_DIR}")

# Quick test on Lysozyme
if test_msa_str:
    test_msa = msa_str_to_pyhmmer(test_msa_str)
    test_hmm = build_hmm(test_msa, name="P00698_lysozyme")
    print(f"\nTest HMM built successfully: M={test_hmm.M} match states")

In [ ]:
#@title Profile HMM forward algorithm — prefix-aware drafter. { display-mode: "form" }

class ProfileHMMDrafter:
    """Prefix-aware drafter using a Profile HMM.

    At each spec-decode step it:
    1. Runs the HMM forward algorithm on the already-accepted prefix to get
       the posterior probability distribution over match states at the current
       position.
    2. Uses those posterior weights to compute the emission distribution at
       the next K positions — a position-specific and prefix-conditioned q(x).

    This is fundamentally different from the CNN: the CNN always produces the
    same draft regardless of what was accepted so far; the HMM sharpens its
    guess based on how well the accepted tokens fit into each match state path.
    """

    def __init__(self, hmm: pyhmmer.plan7.HMM, query_aa: str):
        """
        Args:
            hmm: Profile HMM built from the protein family MSA
            query_aa: The query protein's AA sequence (used for length alignment)
        """
        self.hmm = hmm
        self.M = hmm.M  # number of match states
        self.L = len(query_aa)

        # Extract emission log-probabilities: shape (M, 20)
        # hmm.mat[i] gives match-state i emission log probs over the alphabet
        self.emit_logprobs = np.array([hmm.mat[i] for i in range(1, self.M + 1)])  # (M, 20)

        # Position mapping: query position t (0-indexed) -> HMM match state (1-indexed)
        # For a pruned/aligned HMM this is a direct mapping if M == L
        # For a family HMM (M != L), we align using hmmer's own aligner
        self._build_position_map(query_aa)

        # Forward algorithm state: log-sum-exp forward variables
        # alpha[t][m] = log P(observations 1..t, state at t is match-state m)
        self._log_alpha = None   # will be computed lazily
        self._prefix_len = 0     # length of prefix used to compute _log_alpha

        # Transition log-probabilities (for the forward algorithm)
        # Using simplified model: only match->match transitions
        # hmm.trans[i] = [MM, MI, MD, IM, II, DM, DD] for state i
        # We use MM (match -> match) and begin probabilities
        self._log_trans_mm = np.array([hmm.trans[i][0] for i in range(1, self.M)])  # (M-1,)
        self._log_begin = np.array([hmm.trans[0][i] for i in range(self.M)])  # approx begin probs

    def _build_position_map(self, query_aa: str):
        """Map query positions to HMM match states via Viterbi alignment."""
        if self.M == self.L:
            # Perfect length match — direct 1:1 mapping
            self.pos_to_state = list(range(self.M))  # pos i -> match state i
            return

        # Align query sequence to HMM using pyhmmer
        try:
            bg = Background(ALPHABET)
            pipeline = pyhmmer.plan7.Pipeline(ALPHABET, background=bg)
            query_seq = pyhmmer.easel.TextSequence(
                name=b"query", sequence=query_aa
            ).digitize(ALPHABET)
            hits = pipeline.search_hmm(self.hmm, [query_seq])

            if hits.hits:
                aln = hits.hits[0].domains[0].alignment
                # Build pos -> state map from the alignment
                self.pos_to_state = []
                q_pos = 0
                for q_char, h_char in zip(aln.target_sequence, aln.hmm_sequence):
                    if q_char != '-' and h_char != '-':
                        # Match: query position maps to this HMM column
                        hmm_col = aln.hmm_from + len([c for c in aln.hmm_sequence[:len(self.pos_to_state)] if c != '-']) - 1
                        self.pos_to_state.append(min(hmm_col, self.M - 1))
                        q_pos += 1
                # Fallback if alignment incomplete
                while len(self.pos_to_state) < self.L:
                    self.pos_to_state.append(self.pos_to_state[-1] if self.pos_to_state else 0)
            else:
                # No hit — uniform fallback mapping
                print(f"  WARNING: No HMM alignment found, using uniform position mapping")
                step = self.M / self.L
                self.pos_to_state = [min(int(i * step), self.M - 1) for i in range(self.L)]
        except Exception as e:
            print(f"  WARNING: HMM alignment failed ({e}), using uniform mapping")
            step = self.M / self.L
            self.pos_to_state = [min(int(i * step), self.M - 1) for i in range(self.L)]

    def _aa_to_idx(self, aa_char: str) -> int:
        """Convert AA letter to index in AA_LETTERS (0-19)."""
        return AA_LETTERS.index(aa_char) if aa_char in AA_LETTERS else -1

    def forward_update(self, accepted_prefix: list[str]):
        """Run forward algorithm on accepted prefix to get posterior state distribution.

        Uses a simplified match-only HMM (ignoring insert/delete states) for speed.
        The posterior gives P(being in match state m | prefix observed so far).

        Args:
            accepted_prefix: list of AA characters already accepted by verifier

        Returns:
            log_posterior: (M,) log-probabilities over match states at current position
        """
        if len(accepted_prefix) == 0:
            # At start: use begin probabilities
            return self._log_begin.copy()

        M = self.M
        NEG_INF = -1e30

        # Initialize: alpha at position 0 = begin_prob * emit(obs[0])
        obs_idx = self._aa_to_idx(accepted_prefix[0])
        if obs_idx < 0:
            log_alpha = self._log_begin.copy()
        else:
            log_alpha = self._log_begin + self.emit_logprobs[:, obs_idx]

        # Propagate forward for each subsequent position
        for t in range(1, len(accepted_prefix)):
            obs_idx = self._aa_to_idx(accepted_prefix[t])
            if obs_idx < 0:
                continue  # skip unknown residues

            # log_alpha_new[m] = log_sum_exp over m' of (log_alpha[m'] + log_trans_mm[m']) + emit[m, obs]
            # Vectorized: shift and add transition, then add emission
            log_alpha_prev = log_alpha[:-1] + self._log_trans_mm  # (M-1,)
            log_alpha_new = np.full(M, NEG_INF)
            log_alpha_new[1:] = log_alpha_prev + self.emit_logprobs[1:, obs_idx]
            # Also allow staying in first state (begin -> state 0)
            log_alpha_new[0] = self._log_begin[0] + self.emit_logprobs[0, obs_idx]
            log_alpha = log_alpha_new

        # Normalize to get posterior (subtract log-sum-exp)
        max_val = log_alpha.max()
        if max_val > NEG_INF:
            log_posterior = log_alpha - (max_val + np.log(np.exp(log_alpha - max_val).sum()))
        else:
            log_posterior = np.full(M, -np.log(M))  # uniform fallback

        return log_posterior

    def get_draft_logits(self, pos: int, accepted_prefix: list,
                          encoder_hidden=None) -> torch.Tensor:
        """Return draft logits for positions pos..pos+K as a (k, 20) tensor.

        For each position t = pos..pos+k-1:
        - Get the HMM match state m = pos_to_state[t]
        - Weight by the posterior probability of being near that state
          (from the forward algorithm on the accepted prefix)
        - Return the (weighted) emission log-probabilities as logits

        Args:
            pos: current position (number of tokens generated so far)
            accepted_prefix: list of AA chars already accepted (length == pos)
            encoder_hidden: ignored (HMM doesn't use encoder embeddings)

        Returns:
            Tensor of shape (k, 20) with draft logits in AA_LETTERS order
        """
        K_remaining = self.L - pos
        if K_remaining <= 0:
            return torch.zeros(0, 20)

        # Compute posterior over match states given accepted prefix
        accepted_aa = [AA_LETTERS[AA_TOKEN_ID_TO_CNN_IDX[t]]
                       for t in accepted_prefix
                       if t in AA_TOKEN_ID_TO_CNN_IDX]
        log_posterior = self.forward_update(accepted_aa)  # (M,)

        # For each of the next K positions, compute emission distribution
        # weighted by the posterior over states at that position
        logits = []
        for t in range(pos, min(pos + 16, self.L)):  # max K=16
            # Primary state for this position
            state_idx = min(self.pos_to_state[t], self.M - 1)

            # Soft weighting: average emission probs over a window of states
            # centered on state_idx, weighted by posterior
            window = 3
            lo = max(0, state_idx - window)
            hi = min(self.M, state_idx + window + 1)
            window_states = np.arange(lo, hi)

            # Posterior weights for this window (renormalized)
            window_log_post = log_posterior[window_states]
            window_log_post -= window_log_post.max()
            window_weights = np.exp(window_log_post)
            window_weights /= window_weights.sum()

            # Weighted emission log-probs
            weighted_emit = (window_weights[:, None] * self.emit_logprobs[window_states]).sum(axis=0)
            logits.append(weighted_emit)

        return torch.tensor(np.array(logits), dtype=torch.float32, device=device)


print("ProfileHMMDrafter defined.")

# Quick test
if 'test_hmm' in dir() and test_hmm is not None:
    test_uid = "P00698"
    if test_uid in test_set:
        drafter = ProfileHMMDrafter(test_hmm, test_set[test_uid]["aa"])
        print(f"Drafter created: M={drafter.M}, L={drafter.L}, "
              f"states={drafter.pos_to_state[:5]}...")
        logits = drafter.get_draft_logits(0, [], None)
        print(f"Draft logits shape at pos=0: {logits.shape}")
        top_aa = AA_LETTERS[logits[0].argmax().item()]
        print(f"Top draft AA at position 0: {top_aa}")
    else:
        print("P00698 not in test_set (run baseline notebook first to build FASTAs)")

In [ ]:
#@title HMM drafter: correctness test + integrate into benchmark. { display-mode: "form" }

def build_hmm_drafters_for_test_set(test_set: dict) -> dict[str, ProfileHMMDrafter | None]:
    """Build or load HMM drafters for all proteins in the test set.

    Returns dict: uid -> ProfileHMMDrafter (or None if no Pfam family found).
    """
    drafters = {}
    n_found = 0
    for uid, rec in sorted(test_set.items(), key=lambda kv: kv[1]["length"]):
        hmm = get_hmm_for_protein(uid, rec["aa"])
        if hmm is not None:
            drafters[uid] = ProfileHMMDrafter(hmm, rec["aa"])
            n_found += 1
        else:
            drafters[uid] = None
    print(f"\nHMM drafters built: {n_found}/{len(test_set)} proteins have Pfam families")
    return drafters


# Correctness test: greedy spec-decode with HMM drafter == plain enc-dec
# (The verification loop is identical; only the draft distribution changes.
#  Correctness is guaranteed by the accept/reject rule regardless of drafter quality.)
print("HMM drafter correctness is guaranteed by the spec-decode loop structure:")
print("  - If HMM draft is wrong → verifier's greedy token is used instead")
print("  - Output is ALWAYS the verifier's greedy sequence, regardless of drafter quality")
print("  - The drafter only affects SPEED (acceptance rate), never correctness")
print()

if 'test_hmm' in dir() and test_hmm is not None and "P00698" in test_set:
    uid = "P00698"
    rec = test_set[uid]
    drafter = ProfileHMMDrafter(test_hmm, rec["aa"])

    ref = generate_reference(rec["3di"])
    result = speculative_decode_greedy(
        rec["3di"], K=5,
        get_draft_logits=drafter.get_draft_logits
    )
    match = result["sequence"] == ref
    print(f"Correctness check for {uid} (L={rec['length']}) K=5:")
    print(f"  {'PASS' if match else 'FAIL'}  accept_rate={result['acceptance_rate']:.2f}  "
          f"steps={result['n_steps']}")
    if not match:
        for j, (a, b) in enumerate(zip(ref, result["sequence"])):
            if a != b:
                print(f"  First mismatch at pos {j}: ref={a} hmm={b}")
                break

print()
print("To benchmark HMM drafter across all proteins, the main benchmark loop (Part 4)")
print("needs hmm_drafters dict. Build it with:")
print("  hmm_drafters = build_hmm_drafters_for_test_set(test_set)")
print("Then in the loop, pass get_draft_logits=hmm_drafters[uid].get_draft_logits")
print("(or None for proteins without a Pfam family, falling back to CNN)")

## Part 3 — HuggingFace `assistant_model` Wrapper

Wraps the CNN as a compatible `PreTrainedModel` so we can use `model.generate(..., assistant_model=cnn_wrapper)`.

In [ ]:
#@title CNNAssistantModel wrapper. { display-mode: "form" }
from transformers import PreTrainedModel, T5Config
from transformers.generation.utils import GenerateDecoderOnlyOutput


class CNNAssistantModel(PreTrainedModel):
    """Wraps enc-CNN as a HuggingFace-compatible assistant model for speculative decoding.
    
    The CNN is prefix-independent: it always produces the same logits for a given
    encoder input regardless of what was previously generated. We exploit this by
    pre-computing all logits once and serving slices on demand.
    """
    config_class = T5Config
    
    def __init__(self, config, prostt5_encoder, cnn_head, tok, aa_token_ids):
        super().__init__(config)
        self._encoder = prostt5_encoder
        self._cnn = cnn_head
        self._tokenizer = tok
        self._aa_token_ids = aa_token_ids
        self.config.is_encoder_decoder = True
        self.config.decoder_start_token_id = config.decoder_start_token_id
        self.generation_config = GenerationConfig(
            num_assistant_tokens=5,
            num_assistant_tokens_schedule="constant",
            do_sample=False,
            max_length=3000,
            return_dict_in_generate=True,
            output_scores=True,
        )
        self._cached_logits = None
        self._cached_input_hash = None
    
    def _compute_cnn_logits(self, encoder_outputs):
        """Run CNN on encoder hidden states, return full-vocab logits for all positions."""
        hidden = encoder_outputs[0]  # (1, seq_len, 1024)
        h = hidden[:, 1:-1, :]  # trim prefix + EOS
        logits_20 = self._cnn(h.float())[0]  # (L, 20)
        # Map to full vocab: set all non-AA positions to -inf
        vocab_size = self.config.vocab_size
        full_logits = torch.full((logits_20.shape[0], vocab_size), -100.0,
                                 device=logits_20.device)
        for i, tid in enumerate(self._aa_token_ids):
            full_logits[:, tid] = logits_20[:, i]
        return full_logits
    
    def get_encoder(self):
        return self._encoder
    
    def prepare_inputs_for_generation(self, decoder_input_ids, encoder_outputs=None, **kwargs):
        return {
            "decoder_input_ids": decoder_input_ids,
            "encoder_outputs": encoder_outputs,
        }
    
    def forward(self, decoder_input_ids=None, encoder_outputs=None, **kwargs):
        """Return pre-computed CNN logits for the positions being queried."""
        if encoder_outputs is not None:
            # Compute or use cached logits
            hidden_id = id(encoder_outputs[0])
            if self._cached_input_hash != hidden_id:
                self._cached_logits = self._compute_cnn_logits(encoder_outputs)
                self._cached_input_hash = hidden_id
        
        # decoder_input_ids shape: (1, seq_len)
        seq_len = decoder_input_ids.shape[1]
        # We need to return logits for each position in decoder_input_ids
        # Position i predicts position i+1, so we return logits for positions 0..seq_len-1
        # which predict AA positions 0..seq_len-1 (after decoder_start)
        n_positions = seq_len
        if self._cached_logits is not None:
            L_cached = self._cached_logits.shape[0]
            n_to_return = min(n_positions, L_cached)
            logits = self._cached_logits[:n_to_return].unsqueeze(0)  # (1, n, vocab)
            # Pad if needed
            if n_to_return < n_positions:
                pad = torch.full((1, n_positions - n_to_return, logits.shape[-1]),
                                -100.0, device=logits.device)
                logits = torch.cat([logits, pad], dim=1)
        else:
            logits = torch.zeros((1, n_positions, self.config.vocab_size), device=device)
        
        from transformers.modeling_outputs import Seq2SeqLMOutput
        return Seq2SeqLMOutput(logits=logits)


# Instantiate
assistant_config = model.config
cnn_assistant = CNNAssistantModel(
    config=assistant_config,
    prostt5_encoder=encoder,
    cnn_head=cnn,
    tok=tokenizer,
    aa_token_ids=AA_TOKEN_IDS,
).to(device).eval()

print(f"CNN assistant model created. is_encoder_decoder={cnn_assistant.config.is_encoder_decoder}")
print(f"num_assistant_tokens={cnn_assistant.generation_config.num_assistant_tokens}")

In [ ]:
#@title Test HF assisted generation on a single protein. { display-mode: "form" }

# Pick shortest protein for quick test
test_uid, test_rec = sorted(test_set.items(), key=lambda kv: kv[1]["length"])[0]
print(f"Testing HF assisted generation on {test_uid} (L={test_rec['length']})")

enc_input = tokenizer(
    [_format_3di(test_rec["3di"])], add_special_tokens=True, return_tensors="pt"
).to(device)

# Reference (plain greedy)
ref = generate_reference(test_rec["3di"])
print(f"Reference (plain enc-dec): {ref[:50]}...")

# Try HF assisted generation
try:
    hf_out = model.generate(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
        max_length=test_rec["length"] + 2,
        do_sample=False,
        num_beams=1,
        assistant_model=cnn_assistant,
    )
    hf_result = _decode_aa(hf_out[0])[:test_rec["length"]]
    match = hf_result == ref
    print(f"HF assisted result:       {hf_result[:50]}...")
    print(f"Match with reference: {match}")
    if not match:
        print("NOTE: HF wrapper may need adjustments. Custom implementation is primary.")
except Exception as e:
    print(f"HF assisted generation failed: {e}")
    print("This is expected if the wrapper needs further adaptation.")
    print("The custom implementation (Part 1) is the primary approach.")

## Part 4 — Benchmarking

Measure real speedup across all proteins and K values. Compare to baseline enc-dec timing and α-predicted speedup.

In [ ]:
#@title Timing helpers. { display-mode: "form" }

def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def _reset_peak_mem():
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)


def _peak_mem_gb() -> float:
    if device.type != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated(device) / 1e9


@torch.inference_mode()
def time_encdec(three_di: str) -> float:
    """Time plain enc-dec greedy generation. Returns median wall time."""
    L = len(three_di)
    enc = tokenizer([_format_3di(three_di)], add_special_tokens=True, return_tensors="pt").to(device)
    gen_kwargs = dict(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
        max_length=L+2, do_sample=False, num_beams=1,
    )
    for _ in range(NUM_WARMUP):
        model.generate(**gen_kwargs)
    _sync()
    times = []
    for _ in range(NUM_REPEATS):
        _sync()
        t0 = time.perf_counter()
        model.generate(**gen_kwargs)
        _sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


def time_spec_decode(three_di: str, K: int,
                     get_draft_logits=None,
                     dynamic_k: bool = False,
                     K_init: int = 5, K_min: int = 1, K_max: int = 16) -> tuple[float, dict]:
    """Time speculative decoding with any drafter. Returns (median_wall_time, last_result_dict)."""
    kwargs = dict(K=K, get_draft_logits=get_draft_logits,
                  dynamic_k=dynamic_k, K_init=K_init, K_min=K_min, K_max=K_max)
    for _ in range(NUM_WARMUP):
        speculative_decode_greedy(three_di, **kwargs)
    _sync()
    times = []
    last_result = None
    for _ in range(NUM_REPEATS):
        result = speculative_decode_greedy(three_di, **kwargs)
        times.append(result["wall_time"])
        last_result = result
    return statistics.median(times), last_result


print("Timing helpers defined.")

In [ ]:
#@title Checkpointing. { display-mode: "form" }

def save_spec_checkpoint(state: dict):
    state["timestamp"] = datetime.now().isoformat()
    path = CHECKPOINT_DIR / "spec_decode_checkpoint.pkl"
    with open(path, "wb") as f:
        pickle.dump(state, f)


def load_spec_checkpoint() -> dict | None:
    path = CHECKPOINT_DIR / "spec_decode_checkpoint.pkl"
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return None


print(f"Spec-decode checkpoint dir: {CHECKPOINT_DIR}")

In [ ]:
#@title Main benchmark loop — CNN + HMM drafters. { display-mode: "form" }

# ── Build HMM drafters (fetches Pfam MSAs, ~2-5 min for 100 proteins) ─────────
print("Building HMM drafters for test set...")
print("Proteins without a Pfam family will fall back to CNN drafter.\n")
hmm_drafters = build_hmm_drafters_for_test_set(test_set)
n_hmm = sum(1 for v in hmm_drafters.values() if v is not None)
print(f"\nHMM coverage: {n_hmm}/{len(test_set)} proteins")

# ── Load or initialize checkpoint ─────────────────────────────────────────────
ckpt = load_spec_checkpoint()
if ckpt is not None:
    results = ckpt["results"]
    completed = set(ckpt["completed"])
    print(f"Resuming: {len(completed)} tasks done.")
else:
    results = []
    completed = set()

# ── Drafters to benchmark ──────────────────────────────────────────────────────
DRAFTERS = [
    ("cnn", None),   # enc-CNN, prefix-independent
    ("hmm", "hmm"),  # Profile HMM, prefix-aware (looked up per protein)
]

# K=-1 is the sentinel for dynamic K rows (avoids mixed int/str column type)
K_DYNAMIC = -1

sorted_proteins = sorted(test_set.items(), key=lambda kv: kv[1]["length"])
n_dynamic = len(DRAFTERS)
total_tasks = len(sorted_proteins) * (1 + len(DRAFTERS) * len(K_VALUES) + n_dynamic)
print(f"Total tasks: {len(sorted_proteins)} proteins × "
      f"({len(DRAFTERS)} drafters × {len(K_VALUES)} K + {n_dynamic} dynamic + 1 enc-dec) = {total_tasks}")
print(f"Already done: {len(completed)}, Remaining: {total_tasks - len(completed)}")

t_global_start = time.time()

for uid, rec in sorted_proteins:
    L = rec["length"]
    three_di = rec["3di"]

    # ── enc-dec baseline (once per protein) ───────────────────────────────────
    encdec_key = f"{uid}_encdec"
    if encdec_key not in completed:
        t_enc = time_encdec(three_di)
        results.append({
            "protein_id": uid, "length": L, "drafter": "enc_dec",
            "K": 0, "wall_s": t_enc, "speedup": 1.0,
            "acceptance_rate": None, "mean_tokens_per_step": 1.0, "n_steps": L,
        })
        completed.add(encdec_key)
        print(f"\n[{uid} L={L}]  enc-dec: {t_enc:.2f}s")
    else:
        t_enc = next((r["wall_s"] for r in results
                      if r["protein_id"] == uid and r["drafter"] == "enc_dec"), None)

    # ── Spec-decode: each drafter × each K + one dynamic run ──────────────────
    for drafter_name, drafter_fn in DRAFTERS:
        # Resolve HMM drafter for this protein
        if drafter_fn == "hmm":
            drafter_obj = hmm_drafters.get(uid)
            if drafter_obj is None:
                continue
            fn = drafter_obj.get_draft_logits
        else:
            fn = None

        for K in K_VALUES:
            task_key = f"{uid}_{drafter_name}_K{K}"
            if task_key in completed:
                continue

            t_spec, spec_result = time_spec_decode(three_di, K=K, get_draft_logits=fn)
            speedup = t_enc / t_spec if t_spec > 0 else float("nan")
            results.append({
                "protein_id": uid, "length": L, "drafter": drafter_name,
                "K": K, "wall_s": t_spec, "speedup": speedup,
                "acceptance_rate": spec_result["acceptance_rate"],
                "mean_tokens_per_step": spec_result["mean_tokens_per_step"],
                "n_steps": spec_result["n_steps"],
            })
            completed.add(task_key)
            print(f"  {drafter_name} K={K:2d}: {t_spec:.2f}s  "
                  f"speedup={speedup:.2f}x  accept={spec_result['acceptance_rate']:.2f}")

        # ── Dynamic K run (one per drafter, K_DYNAMIC=-1 sentinel) ────────────
        dyn_task_key = f"{uid}_{drafter_name}_dynamic"
        if dyn_task_key not in completed:
            t_dyn, dyn_result = time_spec_decode(
                three_di, K=5, get_draft_logits=fn,
                dynamic_k=True, K_init=5, K_min=1, K_max=16,
            )
            speedup_dyn = t_enc / t_dyn if t_dyn > 0 else float("nan")
            results.append({
                "protein_id": uid, "length": L, "drafter": drafter_name,
                "K": K_DYNAMIC, "wall_s": t_dyn, "speedup": speedup_dyn,
                "acceptance_rate": dyn_result["acceptance_rate"],
                "mean_tokens_per_step": dyn_result["mean_tokens_per_step"],
                "n_steps": dyn_result["n_steps"],
                "k_history": dyn_result["k_history"],
            })
            completed.add(dyn_task_key)
            mean_k = np.mean(dyn_result["k_history"]) if dyn_result["k_history"] else 0
            print(f"  {drafter_name} K=dyn: {t_dyn:.2f}s  "
                  f"speedup={speedup_dyn:.2f}x  accept={dyn_result['acceptance_rate']:.2f}  "
                  f"mean_K={mean_k:.1f}")

    # Checkpoint after each protein
    save_spec_checkpoint({"results": results, "completed": list(completed)})

    elapsed = time.time() - t_global_start
    done = len(completed)
    remaining = total_tasks - done
    if done > 0:
        eta_min = (elapsed / done) * remaining / 60
        print(f"  ETA: ~{eta_min:.0f} min")

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"BENCHMARK COMPLETE")

## Part 5 — Analysis & Plots

In [ ]:
#@title Aggregate results. { display-mode: "form" }
import pandas as pd

df = pd.DataFrame(results)
df.to_csv(RESULTS_DIR / "spec_decode_results.csv", index=False)

encdec_df = df[df["drafter"] == "enc_dec"]
spec_df    = df[df["drafter"] != "enc_dec"]
fixed_df   = spec_df[spec_df["K"] != K_DYNAMIC]   # K in {1,3,5,8,16}
dyn_df     = spec_df[spec_df["K"] == K_DYNAMIC]    # dynamic K runs

print("=== Speedup by drafter and K ===")
print(f"{'Drafter':>8s}  {'K':>7s}  {'Mean Speedup':>12s}  {'Median':>8s}  {'Accept':>8s}  {'Tok/step':>8s}")
print("-" * 65)
for drafter in ["cnn", "hmm"]:
    for K in K_VALUES:
        subset = fixed_df[(fixed_df["drafter"] == drafter) & (fixed_df["K"] == K)]
        if len(subset) > 0:
            print(f"{drafter:>8s}  {K:7d}  "
                  f"{subset['speedup'].mean():12.2f}  "
                  f"{subset['speedup'].median():8.2f}  "
                  f"{subset['acceptance_rate'].mean():8.3f}  "
                  f"{subset['mean_tokens_per_step'].mean():8.2f}")
    # Dynamic row
    dyn_sub = dyn_df[dyn_df["drafter"] == drafter]
    if len(dyn_sub) > 0:
        mean_k_vals = dyn_sub["k_history"].apply(lambda h: np.mean(h) if isinstance(h, list) and h else float("nan"))
        print(f"{drafter:>8s}  {'dynamic':>7s}  "
              f"{dyn_sub['speedup'].mean():12.2f}  "
              f"{dyn_sub['speedup'].median():8.2f}  "
              f"{dyn_sub['acceptance_rate'].mean():8.3f}  "
              f"{dyn_sub['mean_tokens_per_step'].mean():8.2f}  "
              f"(mean_K={mean_k_vals.mean():.1f})")

print(f"\nCNN proteins (fixed K): {fixed_df[fixed_df['drafter']=='cnn']['protein_id'].nunique()}")
print(f"HMM proteins (fixed K): {fixed_df[fixed_df['drafter']=='hmm']['protein_id'].nunique()} "
      f"(only proteins with Pfam families)")
print(f"Dynamic K runs — CNN: {dyn_df[dyn_df['drafter']=='cnn']['protein_id'].nunique()}, "
      f"HMM: {dyn_df[dyn_df['drafter']=='hmm']['protein_id'].nunique()}")

In [ ]:
#@title Compare predicted (from α) vs. measured speedup. { display-mode: "form" }

if alpha_predictions:
    comparison_rows = []
    for uid in spec_df["protein_id"].unique():
        if uid not in alpha_predictions:
            continue
        alpha_data = alpha_predictions[uid]
        # Use greedy T=1.0 alpha
        greedy_alpha = alpha_data.get("greedy_T1.0", {}).get("alpha_mean", None)
        if greedy_alpha is None:
            continue
        
        for K in K_VALUES:
            measured = spec_df[(spec_df["protein_id"] == uid) & (spec_df["K"] == K)]
            if len(measured) == 0:
                continue
            
            # Predicted tokens per step from Theorem 3.8
            a = greedy_alpha
            if a < 1.0:
                predicted_tok_per_step = (1 - a**(K+1)) / (1 - a)
            else:
                predicted_tok_per_step = K + 1
            
            comparison_rows.append({
                "protein_id": uid,
                "K": K,
                "alpha": a,
                "predicted_tok_per_step": predicted_tok_per_step,
                "measured_tok_per_step": measured["mean_tokens_per_step"].values[0],
                "measured_speedup": measured["speedup"].values[0],
                "measured_acceptance": measured["acceptance_rate"].values[0],
            })
    
    comp_df = pd.DataFrame(comparison_rows)
    comp_df.to_csv(RESULTS_DIR / "predicted_vs_measured.csv", index=False)
    
    print("=== Predicted vs. Measured (tokens/step) ===")
    print(f"{'K':>3s}  {'Predicted':>10s}  {'Measured':>10s}  {'Ratio':>7s}")
    print("-" * 35)
    for K in K_VALUES:
        subset = comp_df[comp_df["K"] == K]
        if len(subset) > 0:
            pred = subset["predicted_tok_per_step"].mean()
            meas = subset["measured_tok_per_step"].mean()
            print(f"{K:3d}  {pred:10.2f}  {meas:10.2f}  {meas/pred:7.2f}")
    
    print(f"\nIf ratio < 1.0: CNN's prefix-independence hurts acceptance in practice.")
    print(f"If ratio ~ 1.0: α prediction is accurate.")
else:
    print("No alpha predictions available. Run baseline notebook first.")
    comp_df = pd.DataFrame()

In [ ]:
#@title Plots: CNN vs HMM drafter comparison. { display-mode: "form" }
import matplotlib.pyplot as plt

COLORS = {"cnn": "steelblue", "hmm": "darkorange"}
MARKERS_DYN = {"cnn": "s", "hmm": "D"}
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Speedup vs K — CNN vs HMM, fixed + dynamic
ax = axes[0, 0]
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[fixed_df["drafter"] == drafter]
    by_k = sub.groupby("K")["speedup"].median()
    ax.plot(by_k.index, by_k.values, "o-", label=drafter, color=COLORS[drafter])
    # Dynamic K point plotted at x = mean_K for that drafter
    dyn_sub = dyn_df[dyn_df["drafter"] == drafter]
    if len(dyn_sub) > 0:
        dyn_mean_k = dyn_sub["k_history"].apply(lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")).mean()
        dyn_speedup = dyn_sub["speedup"].median()
        ax.scatter([dyn_mean_k], [dyn_speedup], marker=MARKERS_DYN[drafter],
                   color=COLORS[drafter], s=80, zorder=5,
                   label=f"{drafter} dynamic (mean_K={dyn_mean_k:.1f})")
ax.axhline(1.0, color="red", linestyle="--", alpha=0.4, label="breakeven")
ax.set_xlabel("Draft length K")
ax.set_ylabel("Median speedup over enc-dec")
ax.set_title("Speedup vs. K")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,1) Acceptance rate vs K — CNN vs HMM, fixed + dynamic
ax = axes[0, 1]
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[fixed_df["drafter"] == drafter]
    by_k = sub.groupby("K")["acceptance_rate"].mean()
    ax.plot(by_k.index, by_k.values, "o-", label=drafter, color=COLORS[drafter])
    dyn_sub = dyn_df[dyn_df["drafter"] == drafter]
    if len(dyn_sub) > 0:
        dyn_mean_k = dyn_sub["k_history"].apply(lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")).mean()
        ax.scatter([dyn_mean_k], [dyn_sub["acceptance_rate"].mean()],
                   marker=MARKERS_DYN[drafter], color=COLORS[drafter], s=80, zorder=5,
                   label=f"{drafter} dynamic")
ax.set_xlabel("Draft length K")
ax.set_ylabel("Mean acceptance rate")
ax.set_title("Acceptance rate vs. K")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,2) CNN vs HMM acceptance rate scatter (shared proteins, K=5)
ax = axes[0, 2]
K_compare = 5
cnn_k5 = fixed_df[(fixed_df["drafter"] == "cnn") & (fixed_df["K"] == K_compare)].set_index("protein_id")
hmm_k5 = fixed_df[(fixed_df["drafter"] == "hmm") & (fixed_df["K"] == K_compare)].set_index("protein_id")
shared = cnn_k5.index.intersection(hmm_k5.index)
if len(shared) > 0:
    ax.scatter(cnn_k5.loc[shared, "acceptance_rate"], hmm_k5.loc[shared, "acceptance_rate"],
               alpha=0.6, s=20)
    lim = max(cnn_k5.loc[shared, "acceptance_rate"].max(),
              hmm_k5.loc[shared, "acceptance_rate"].max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", alpha=0.4, label="CNN == HMM")
    ax.set_xlabel(f"CNN acceptance rate (K={K_compare})")
    ax.set_ylabel(f"HMM acceptance rate (K={K_compare})")
    ax.set_title("CNN vs HMM acceptance (shared proteins)")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No shared proteins yet", ha="center", va="center", transform=ax.transAxes)

# (1,0) Speedup vs length — best fixed K, both drafters
ax = axes[1, 0]
best_k = fixed_df.groupby("K")["speedup"].median().idxmax() if len(fixed_df) > 0 else 5
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[(fixed_df["drafter"] == drafter) & (fixed_df["K"] == best_k)]
    ax.scatter(sub["length"], sub["speedup"], alpha=0.5, s=15,
               label=drafter, color=COLORS[drafter])
ax.axhline(1.0, color="red", linestyle="--", alpha=0.4)
ax.set_xscale("log")
ax.set_xlabel("Protein length")
ax.set_ylabel(f"Speedup (K={best_k})")
ax.set_title(f"Speedup vs. length (best fixed K={best_k})")
ax.legend()
ax.grid(True, alpha=0.3)

# (1,1) Latency: enc-dec vs CNN vs HMM (best fixed K)
ax = axes[1, 1]
ax.scatter(encdec_df["length"], encdec_df["wall_s"], alpha=0.5, s=15,
           label="enc-dec", color="gray")
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[(fixed_df["drafter"] == drafter) & (fixed_df["K"] == best_k)]
    ax.scatter(sub["length"], sub["wall_s"], alpha=0.5, s=15,
               label=f"spec {drafter} K={best_k}", color=COLORS[drafter])
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Protein length")
ax.set_ylabel("Wall time (s)")
ax.set_title("Latency comparison")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# (1,2) Predicted vs measured (CNN only, needs alpha from notebook 1)
ax = axes[1, 2]
if len(comp_df) > 0:
    ax.scatter(comp_df["predicted_tok_per_step"], comp_df["measured_tok_per_step"],
               alpha=0.4, s=15)
    max_val = max(comp_df["predicted_tok_per_step"].max(), comp_df["measured_tok_per_step"].max())
    ax.plot([0, max_val], [0, max_val], "r--", alpha=0.5, label="perfect prediction")
    ax.set_xlabel("Predicted tokens/step (α, Theorem 3.8)")
    ax.set_ylabel("Measured tokens/step")
    ax.set_title("Predicted vs. measured (CNN)")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "Run baseline notebook first\nfor α predictions",
            ha="center", va="center", transform=ax.transAxes)

fig.suptitle("ProstT5 Speculative Decoding: CNN vs Profile HMM Drafter", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "spec_decode_plots.png", dpi=150)
plt.show()
print(f"Saved to {RESULTS_DIR / 'spec_decode_plots.png'}")

In [ ]:
#@title Overhead analysis. { display-mode: "form" }

print("=== Overhead Analysis ===")
print(f"\nDraft model: enc-CNN (prefix-independent, one-shot)")
print(f"CNN params: {sum(p.numel() for p in cnn.parameters()):,} (vs ProstT5: {sum(p.numel() for p in model.parameters()):,})")
print(f"CNN overhead: single forward pass, negligible vs decoder verification")

# Measure CNN-only time on a few proteins
print(f"\n--- CNN drafting time (one-shot, all L positions) ---")
for uid, rec in sorted(test_set.items(), key=lambda kv: kv[1]["length"])[:5]:
    enc_input = tokenizer([_format_3di(rec["3di"])], add_special_tokens=True, return_tensors="pt").to(device)
    _sync()
    t0 = time.perf_counter()
    with torch.inference_mode():
        h = encoder(input_ids=enc_input.input_ids, attention_mask=enc_input.attention_mask).last_hidden_state
        h = h[:, 1:-1, :]
        _ = cnn(h.float())
    _sync()
    cnn_time = time.perf_counter() - t0
    print(f"  {uid} L={rec['length']:4d}: CNN forward = {cnn_time*1000:.1f}ms")

print(f"\nNote: encoder forward is shared between CNN and decoder verification,")
print(f"so CNN overhead is only the CNN head itself (< 1ms for any length).")

# vRAM comparison
if device.type == "cuda":
    _reset_peak_mem()
    uid, rec = sorted(test_set.items(), key=lambda kv: kv[1]["length"])[-1]  # longest
    _ = speculative_decode_greedy(rec["3di"], K=5)
    spec_vram = _peak_mem_gb()
    print(f"\nPeak vRAM during spec decode (longest protein, K=5): {spec_vram:.2f} GB")

In [ ]:
#@title Dynamic K behavior analysis. { display-mode: "form" }

if len(dyn_df) == 0:
    print("No dynamic K results yet.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # (0) Distribution of mean_K across proteins, per drafter
    ax = axes[0]
    for drafter in ["cnn", "hmm"]:
        sub = dyn_df[dyn_df["drafter"] == drafter]
        if len(sub) == 0:
            continue
        mean_ks = sub["k_history"].apply(
            lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
        ).dropna()
        ax.hist(mean_ks, bins=15, alpha=0.6, label=drafter, color=COLORS[drafter])
    ax.axvline(5, color="gray", linestyle="--", alpha=0.5, label="K_init=5")
    ax.set_xlabel("Mean K over sequence")
    ax.set_ylabel("Number of proteins")
    ax.set_title("Distribution of mean dynamic K")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (1) K trace for 6 representative proteins (3 high-α, 3 low-α), CNN drafter
    ax = axes[1]
    cnn_dyn = dyn_df[dyn_df["drafter"] == "cnn"].copy()
    cnn_dyn["mean_k"] = cnn_dyn["k_history"].apply(
        lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
    )
    cnn_dyn_sorted = cnn_dyn.dropna(subset=["mean_k"]).sort_values("mean_k")
    # Pick 3 lowest and 3 highest mean_K proteins
    representatives = pd.concat([cnn_dyn_sorted.head(3), cnn_dyn_sorted.tail(3)])
    cmap = plt.cm.RdYlGn
    for i, (_, row) in enumerate(representatives.iterrows()):
        hist = row["k_history"]
        if not isinstance(hist, list) or len(hist) == 0:
            continue
        color = cmap(i / max(len(representatives) - 1, 1))
        label = f"{row['protein_id']} (L={row['length']}, mean_K={row['mean_k']:.1f})"
        ax.plot(hist, alpha=0.8, linewidth=1, color=color, label=label)
    ax.axhline(5, color="gray", linestyle="--", alpha=0.4, label="K_init=5")
    ax.set_xlabel("Decoding step")
    ax.set_ylabel("K used")
    ax.set_title("K trace — 3 easiest & 3 hardest proteins (CNN)")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.3)

    # (2) mean_K vs acceptance rate scatter — does high acceptance → high mean_K?
    ax = axes[2]
    for drafter in ["cnn", "hmm"]:
        sub = dyn_df[dyn_df["drafter"] == drafter].copy()
        if len(sub) == 0:
            continue
        sub["mean_k"] = sub["k_history"].apply(
            lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
        )
        ax.scatter(sub["acceptance_rate"], sub["mean_k"],
                   alpha=0.5, s=15, label=drafter, color=COLORS[drafter])
    ax.axhline(5, color="gray", linestyle="--", alpha=0.4, label="K_init=5")
    ax.set_xlabel("Acceptance rate")
    ax.set_ylabel("Mean K over sequence")
    ax.set_title("Acceptance rate vs. mean dynamic K")
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.suptitle("Dynamic K Behavior Analysis", fontsize=13)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "dynamic_k_analysis.png", dpi=150)
    plt.show()
    print(f"Saved to {RESULTS_DIR / 'dynamic_k_analysis.png'}")

    # Summary stats
    print("\n=== Dynamic K summary ===")
    for drafter in ["cnn", "hmm"]:
        sub = dyn_df[dyn_df["drafter"] == drafter].copy()
        if len(sub) == 0:
            continue
        sub["mean_k"] = sub["k_history"].apply(
            lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
        )
        print(f"\n{drafter.upper()}:")
        print(f"  Mean K (across proteins): {sub['mean_k'].mean():.2f}  "
              f"(K_init=5, range [{sub['mean_k'].min():.1f}, {sub['mean_k'].max():.1f}])")
        print(f"  Speedup: mean={sub['speedup'].mean():.2f}x  median={sub['speedup'].median():.2f}x")
        print(f"  Acceptance rate: {sub['acceptance_rate'].mean():.3f}")

In [ ]:
#@title Save all results to Drive. { display-mode: "form" }

# Save final state
save_spec_checkpoint({"results": results, "completed": list(completed)})

# Copy key outputs to Drive for persistence
if USE_DRIVE:
    import shutil
    drive_results = Path(DRIVE_ROOT) / "spec_decode_results"
    drive_results.mkdir(exist_ok=True)
    for f in RESULTS_DIR.glob("*"):
        if f.is_file():
            shutil.copy(f, drive_results / f.name)
    print(f"Results copied to Drive: {drive_results}")
else:
    print(f"Results at: {RESULTS_DIR}")

print("\nDone! Key files:")
print(f"  - spec_decode_results.csv (all timings)")
print(f"  - predicted_vs_measured.csv (α comparison)")
print(f"  - spec_decode_plots.png")